# Isolation Forest — PaySim Fraud Detection

**Goal:** Test an Isolation Forest anomaly detector on the PaySim dataset.

This notebook is intentionally detailed. Run **one cell at a time** and read the printed output after each cell.

### Important rules
- `isFraud` is used only as the ground-truth label for evaluation.
- `isFlaggedFraud` is not used as a model input.
- Account IDs are excluded because they are high-cardinality identifiers rather than useful numeric behavior features.
- The model is trained only on transactions labelled as legitimate (`isFraud = 0`) in the training split.
- The threshold is selected on the validation set and the final metrics are reported on the untouched test set.

In [ ]:
# BLOCK 1 — Imports and environment check

import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    pass

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

print("=" * 80)
print("BLOCK 1: ENVIRONMENT CHECK")
print("=" * 80)

print("Python / notebook environment loaded successfully.")
print("Pandas version       :", pd.__version__)
print("NumPy version        :", np.__version__)

try:
    import sklearn
    print("Scikit-learn version :", sklearn.__version__)
except Exception as exc:
    print("Could not read scikit-learn version:", exc)

print("Ready for the next block.")


### Output after Block 1

You should see the Python data-science libraries load successfully and their versions.

If this block fails, fix the Python environment before continuing. Do not move to model training until the environment is working.

In [ ]:
# BLOCK 2 — Configuration

DATA_PATH = Path(r".\data\PS_20174392719_1491204439457_log.csv")

# Start with a manageable sample for the first model test.
# Set to None later if you want to experiment with the full dataset.
MAX_ROWS = 500_000

RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE_FROM_REMAINDER = 0.25  # gives 20% validation and 60% train overall

N_ESTIMATORS = 200
CONTAMINATION = "auto"
N_JOBS = -1

MODEL_DIR = Path("models")
RESULT_DIR = Path("results")
MODEL_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

print("=" * 80)
print("BLOCK 2: CONFIGURATION")
print("=" * 80)
print("Dataset path         :", DATA_PATH.resolve())
print("Maximum rows         :", MAX_ROWS if MAX_ROWS is not None else "FULL DATASET")
print("Random state         :", RANDOM_STATE)
print("Train / Validation / Test target split: 60% / 20% / 20%")
print("Isolation Forest trees:", N_ESTIMATORS)
print("Contamination         :", CONTAMINATION)
print("Artifacts directory   :", MODEL_DIR.resolve())
print("Results directory     :", RESULT_DIR.resolve())

### Output after Block 2

Check that the dataset path points to your project's `data` folder.

For the first run, `MAX_ROWS = 500_000` keeps the experiment practical. After the pipeline works, you can increase it or set it to `None` for a larger experiment.

In [ ]:
# BLOCK 3 — Load PaySim data

print("=" * 80)
print("BLOCK 3: LOADING PAYSIM DATA")
print("=" * 80)

if not DATA_PATH.exists():
    fallback_path = Path.cwd() / "data" / "PS_20174392719_1491204439457_log.csv"
    if fallback_path.exists():
        DATA_PATH = fallback_path
    else:
        raise FileNotFoundError(
            f"PaySim file was not found at: {DATA_PATH.resolve()}\n"
            "Check that the CSV is inside the project's data folder."
        )

start = time.perf_counter()

read_kwargs = {
    "low_memory": False,
}

if MAX_ROWS is not None:
    read_kwargs["nrows"] = MAX_ROWS

df = pd.read_csv(DATA_PATH, **read_kwargs)

load_time = time.perf_counter() - start

print(f"Rows loaded           : {len(df):,}")
print(f"Columns loaded        : {len(df.columns)}")
print(f"Load time             : {load_time:.2f} seconds")
print()
print("Columns:")
print(list(df.columns))
print()
print("First 5 rows:")
display(df.head())


### Output after Block 3

You should see the PaySim columns and the first few records.

The important label is `isFraud`. Do not put that column into the model features.

In [ ]:
# BLOCK 4 — Data quality and fraud distribution

print("=" * 80)
print("BLOCK 4: DATA QUALITY CHECK")
print("=" * 80)

print("Missing values:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing.to_frame("missing_count"))

print()
print("Fraud label distribution:")
label_counts = df["isFraud"].value_counts().sort_index()
label_pct = (df["isFraud"].value_counts(normalize=True).sort_index() * 100).round(4)

distribution = pd.DataFrame({
    "count": label_counts,
    "percentage": label_pct
})
display(distribution)

print()
print(f"Fraud transactions   : {int(label_counts.get(1, 0)):,}")
print(f"Legitimate txns      : {int(label_counts.get(0, 0)):,}")
print(f"Fraud rate           : {float(label_pct.get(1, 0)):.4f}%")

### Output after Block 4

This block tells you how imbalanced the dataset is.

Do not be surprised if fraud is a very small minority. That is exactly why accuracy is not our main model-selection metric.

In [ ]:
# BLOCK 5 — Feature engineering

print("=" * 80)
print("BLOCK 5: FEATURE ENGINEERING")
print("=" * 80)

work = df.copy()

# Numeric behavioral features
work["orig_balance_change"] = work["oldbalanceOrg"] - work["newbalanceOrig"]
work["dest_balance_change"] = work["newbalanceDest"] - work["oldbalanceDest"]
work["orig_balance_error"] = work["amount"] - work["orig_balance_change"]
work["dest_balance_error"] = work["amount"] - work["dest_balance_change"]

# Useful binary indicators
work["orig_zero_after"] = (work["newbalanceOrig"] == 0).astype(int)
work["dest_zero_before"] = (work["oldbalanceDest"] == 0).astype(int)
work["dest_zero_after"] = (work["newbalanceDest"] == 0).astype(int)

# Log transform for highly skewed monetary amounts
work["log_amount"] = np.log1p(work["amount"])

# One-hot encode transaction type
work = pd.get_dummies(work, columns=["type"], prefix="type", dtype=int)

# Ensure all 5 expected PaySim transaction types exist as dummy columns
for t in ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]:
    col = f"type_{t}"
    if col not in work.columns:
        work[col] = 0

feature_columns = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "orig_balance_change",
    "dest_balance_change",
    "orig_balance_error",
    "dest_balance_error",
    "orig_zero_after",
    "dest_zero_before",
    "dest_zero_after",
    "log_amount",
]

type_columns = sorted([c for c in work.columns if c.startswith("type_")])
feature_columns += type_columns

X = work[feature_columns].astype(np.float32)
y = work["isFraud"].astype(int)

print("Number of features :", X.shape[1])
print("Features:")
for i, name in enumerate(feature_columns, start=1):
    print(f"{i:02d}. {name}")

print()
print("Feature matrix shape:", X.shape)
print("Target shape        :", y.shape)
display(X.head())


### Output after Block 5

You should see the final numeric feature set.

Notice that:
- `isFraud` is separated into `y`.
- `nameOrig` and `nameDest` are not model features.
- `isFlaggedFraud` is not a model feature.
- Transaction type is converted into numeric one-hot columns.

In [ ]:
# BLOCK 6 — Train / validation / test split

print("=" * 80)
print("BLOCK 6: FAIR DATA SPLIT")
print("=" * 80)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=VALIDATION_SIZE_FROM_REMAINDER,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print(f"Training rows        : {len(X_train):,}")
print(f"Validation rows      : {len(X_val):,}")
print(f"Test rows            : {len(X_test):,}")
print()

for name, labels in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    fraud_count = int(labels.sum())
    fraud_rate = float(labels.mean() * 100)
    print(f"{name:12s} -> fraud={fraud_count:,}, fraud_rate={fraud_rate:.4f}%")

# Important for anomaly detection:
# train only on legitimate examples to reduce contamination.
normal_mask = y_train == 0
X_train_normal = X_train.loc[normal_mask].copy()

print()
print(f"Normal-only training rows: {len(X_train_normal):,}")

### Output after Block 6

The split should be approximately **60% train / 20% validation / 20% test**.

The model itself will see only legitimate training transactions. The validation and test sets remain mixed so we can measure fraud-detection performance.

In [ ]:
# BLOCK 7 — Scale features

print("=" * 80)
print("BLOCK 7: FEATURE SCALING")
print("=" * 80)

scaler = StandardScaler()

X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Training normal matrix :", X_train_normal_scaled.shape)
print("Validation matrix     :", X_val_scaled.shape)
print("Test matrix           :", X_test_scaled.shape)

print()
print("First training row after scaling:")
print(np.round(X_train_normal_scaled[0], 4))

### Output after Block 7

The scaler is fitted **only on legitimate training data**. Validation and test data are transformed using that fitted scaler.

This prevents information from the test set leaking into training.

In [ ]:
# BLOCK 8 — Train Isolation Forest

print("=" * 80)
print("BLOCK 8: TRAINING ISOLATION FOREST")
print("=" * 80)

model = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS
)

start = time.perf_counter()
model.fit(X_train_normal_scaled)
training_time = time.perf_counter() - start

print(f"Training time        : {training_time:.2f} seconds")
print(f"Estimators           : {N_ESTIMATORS}")
print(f"Features             : {X_train_normal_scaled.shape[1]}")
print("Model training status: SUCCESS")

### Output after Block 8

This is the actual anomaly detector training step.

Isolation Forest learns what normal/typical transactions look like by isolating unusual observations.

In [ ]:
# BLOCK 9 — Generate anomaly scores

print("=" * 80)
print("BLOCK 9: SCORING VALIDATION AND TEST DATA")
print("=" * 80)

# decision_function: higher = more normal
# We negate it so higher score = more anomalous.
val_scores = -model.decision_function(X_val_scaled)
test_scores = -model.decision_function(X_test_scaled)

print("Validation score statistics:")
print(pd.Series(val_scores).describe())

print()
print("Test score statistics:")
print(pd.Series(test_scores).describe())

print()
print("Score direction check:")
print("Higher anomaly score => more suspicious transaction")

### Output after Block 9

The anomaly score is continuous. It is not yet a 0/1 fraud prediction.

We still need a threshold: transactions above that threshold become alerts.

In [ ]:
# BLOCK 10 — Select threshold on validation data

print("=" * 80)
print("BLOCK 10: VALIDATION THRESHOLD SELECTION")
print("=" * 80)

thresholds = np.quantile(val_scores, np.linspace(0.90, 0.9999, 300))

best_threshold = None
best_f1 = -1.0
best_precision = 0.0
best_recall = 0.0

for threshold in thresholds:
    val_pred = (val_scores >= threshold).astype(int)

    p = precision_score(y_val, val_pred, zero_division=0)
    r = recall_score(y_val, val_pred, zero_division=0)
    f = f1_score(y_val, val_pred, zero_division=0)

    if f > best_f1:
        best_f1 = f
        best_threshold = float(threshold)
        best_precision = float(p)
        best_recall = float(r)

if best_threshold is None:
    best_threshold = float(np.percentile(val_scores, 95))

print(f"Selected threshold  : {best_threshold:.6f}")
print(f"Validation Precision : {best_precision:.4f}")
print(f"Validation Recall    : {best_recall:.4f}")
print(f"Validation F1        : {best_f1:.4f}")


### Output after Block 10

This threshold is selected **using validation data only**.

The test set has not been used to choose the operating point. That keeps the final test result more trustworthy.

In [ ]:
# BLOCK 11 — Final test evaluation

print("=" * 80)
print("BLOCK 11: FINAL ISOLATION FOREST EVALUATION")
print("=" * 80)

test_pred = (test_scores >= best_threshold).astype(int)

precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, test_scores)
pr_auc = average_precision_score(y_test, test_scores)
cm = confusion_matrix(y_test, test_pred)

tn, fp, fn, tp = cm.ravel()

print(f"Precision            : {precision:.4f}")
print(f"Recall               : {recall:.4f}")
print(f"F1-score             : {f1:.4f}")
print(f"ROC-AUC              : {roc_auc:.4f}")
print(f"PR-AUC                : {pr_auc:.4f}")
print()
print(f"True Negatives (TN)  : {tn:,}")
print(f"False Positives (FP) : {fp:,}")
print(f"False Negatives (FN) : {fn:,}")
print(f"True Positives (TP)  : {tp:,}")
print()
print("Classification report:")
print(classification_report(y_test, test_pred, digits=4, zero_division=0))

print("Confusion matrix:")
display(pd.DataFrame(
    cm,
    index=["Actual Legitimate", "Actual Fraud"],
    columns=["Predicted Legitimate", "Predicted Fraud"]
))

### Output after Block 11

These are the main numbers we will later compare with the Autoencoder.

**Pay particular attention to Recall, F1, PR-AUC, and False Negatives.**

A false negative means a fraudulent transaction was not detected.

In [ ]:
# BLOCK 12 — Visual evaluation

print("=" * 80)
print("BLOCK 12: VISUAL EVALUATION")
print("=" * 80)

fig = plt.figure(figsize=(9, 5))
plt.hist(test_scores[y_test.values == 0], bins=100, alpha=0.65, label="Legitimate")
plt.hist(test_scores[y_test.values == 1], bins=100, alpha=0.65, label="Fraud")
plt.axvline(best_threshold, linestyle="--", linewidth=2, label="Selected threshold")
plt.xlabel("Anomaly score")
plt.ylabel("Number of transactions")
plt.title("Isolation Forest — Test Score Distribution")
plt.legend()
plt.show()

print("The threshold line is the operating point used for the final test predictions.")

In [ ]:
# BLOCK 13 — Save model, scaler, feature list, and results

import joblib

print("=" * 80)
print("BLOCK 13: SAVING ARTIFACTS")
print("=" * 80)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(model, str(MODEL_DIR / "isolation_forest.pkl"))
joblib.dump(scaler, str(MODEL_DIR / "isolation_forest_scaler.pkl"))

with open(MODEL_DIR / "isolation_forest_features.json", "w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)

results = {
    "model": "IsolationForest",
    "rows_loaded": int(len(df)),
    "n_features": int(X.shape[1]),
    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_val)),
    "test_rows": int(len(X_test)),
    "normal_train_rows": int(len(X_train_normal)),
    "training_time_seconds": float(training_time),
    "threshold": float(best_threshold),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
}

with open(RESULT_DIR / "isolation_forest_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Saved:")
print(" - models/isolation_forest.pkl")
print(" - models/isolation_forest_scaler.pkl")
print(" - models/isolation_forest_features.json")
print(" - results/isolation_forest_results.json")
print()
print("Isolation Forest experiment complete.")


## Final note

Do **not** decide the final production model from this notebook alone.

Run the Autoencoder notebook with the same dataset setup and compare both result JSON files. The goal is an evidence-based model comparison rather than assuming one algorithm is better.